# Street Dimension Table - Gold Layer

## Overview
Builds a master street reference dimension by consolidating unique streets from business and closure data. Uses **merge-based incremental updates** to preserve surrogate key stability and referential integrity.

## Purpose
* Create master street reference for dimensional model
* Generate stable, deterministic street IDs for foreign key relationships
* Track which source systems contribute each street
* Support idempotent processing (safe to rerun)

## Key Features
**Merge-based updates** - only new streets inserted, existing preserved  
**Stable surrogate keys** - hash-based IDs never change  
**Multi-source consolidation** - unions business + closure streets  
**Source tracking** - records contributing systems  
**Idempotent** - reruns don't create duplicates or break foreign keys


## Output Schema
**Table**: `workspace.tel_aviv_municipality.dim_street`

| Column | Type | Description |
|--------|------|-------------|
| `street_id` | INT | Hash-based surrogate key (stable across runs) |
| `street_name` | STRING | Canonical street name (cleansed in silver layer) |
| `metadata_source_list` | STRING | Contributing systems (e.g., "API_BUSINESS, CSV_CLOSURES") |
| `ingestion_at` | TIMESTAMP | When street was first added |
| `is_active` | BOOLEAN | Active flag (TRUE = active) |

**Grain**: One row per unique street name


### Key Generation
* `street_id = abs(hash(street_name))`
* **Deterministic**: Same street → same ID every time
* **Stable**: IDs never change, preserving foreign key relationships

### Why Merge-Based?
**Problem with overwrite**: Surrogate keys regenerate → breaks fact table foreign keys  
**Solution with merge**: Surrogate keys preserved → foreign keys remain valid

## Integration
**Upstream**: `stg.business`, `stg.street_closures` 
**Downstream**: `fact_daily_business_compensation`

## Notes
* Street names assumed cleansed in silver layer
* First run bootstraps table, subsequent runs merge incrementally
* Hash collisions theoretically possible but rare
* NULL street names should be filtered in upstream silver layer

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
# Gather unique streets from staging
biz_names = spark.table("workspace.tel_aviv_municipality_stg.business").select(
    F.col("street_name"), 
    F.lit("API_BUSINESS").alias("source_system")
)
closure_names = spark.table("workspace.tel_aviv_municipality_stg.street_closures").select(
    F.col("street_name"), 
    F.lit("CSV_CLOSURES").alias("source_system")
)

# Incremental Update DataFrame
df_source = biz_names.union(closure_names) \
    .groupBy("street_name") \
    .agg(F.array_join(F.collect_set("source_system"), ", ").alias("metadata_source_list")) \
    .withColumn("street_id", F.abs(F.hash(F.col("street_name")))) \
    .withColumn("ingestion_at", F.current_timestamp()) \
    .withColumn("is_active", F.lit(True))

# Merge
destination_table = "workspace.tel_aviv_municipality.dim_street"

if spark.catalog.tableExists(destination_table):
    target_table = DeltaTable.forName(spark, destination_table)
    
    (target_table.alias("t")
     .merge(
         df_source.alias("s"),
         "t.street_id = s.street_id"
     )
     # If street exists, only update metadata if needed (or do nothing to preserve ingestion_at)
     .whenMatchedUpdate(set={"is_active": "s.is_active"}) 
     # If new street, insert it
     .whenNotMatchedInsertAll()
     .execute())
else:
    # For first-time step
    df_source.write.format("delta").mode("overwrite").saveAsTable(destination_table)